# tool_03 双吸附构建（从 1_body/Final 读取 best）

## 目标
- 从 `1_body/Final` 中读取你指定的两个最佳单吸附结构（1 和 2）
- 依据表面两条最小路径方向，生成双吸附候选结构
- 仅生成 `.vasp` + 原子标号，不做计算


In [ ]:
from pathlib import Path
import sys
import tarfile
from fnmatch import fnmatch
import numpy as np
from ase import Atoms
from ase.io import read, write


In [ ]:
# 加载 build/shared
_p = Path.cwd()
while _p != _p.parent:
    if (_p / 'shared' / 'config.py').exists():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent

from shared.load_config import load_config
from shared.single_adsorption_builder import get_species_molecule

cfg = load_config()
go_root = Path(cfg.GO_ROOT)
n_fixed = int(getattr(cfg, 'DEFAULT_N_FIXED', 48))
print(f'GO_ROOT: {go_root}')
print(f'DEFAULT_N_FIXED: {n_fixed}')

In [ ]:
# ---------- 用户参数（先改这里）----------
# 1 和 2 由你指定，1->2 或 2->1 思路完全一致
species_1 = 'OH'
species_2 = 'H'

# 从 1_body/Final 读取最佳单吸附
absorption_root = go_root.parent  # 默认: .../absorption/absorption
single_final_root = absorption_root / '1_body' / 'Final'

# 可显式指定文件；None 则按 species 自动搜索第一个匹配
single_best_file_1 = None
single_best_file_2 = None

# layer 与中心层
layer_from_top = 1
center_layer_from_top = 1

# 路径步长缩放（1.0=最近邻长度）
direction_distance_scale = 1.0

# 输出母 folder：默认 GO_ROOT；可自定义
parent_folder_name = 'GO_ROOT'
export_atom_index = True

# 打包设置
package_vasp = True
package_all_vasp = True
package_patterns = ['*double*.vasp']
package_name = None

assert layer_from_top >= 1 and center_layer_from_top >= 1

In [ ]:
def find_single_best_file(root: Path, species: str, explicit: str | None):
    if explicit:
        p = Path(explicit)
        if not p.is_absolute():
            p = root / p
        if not p.exists():
            raise FileNotFoundError(f'未找到指定文件: {p}')
        return p

    cands = sorted(root.rglob(f'*{species}*.vasp'))
    if not cands:
        raise FileNotFoundError(f'在 {root} 下未找到 {species} 相关 .vasp')
    return cands[0]


def get_layer_cu_positions(slab: Atoms, layer_idx: int, tol: float = 0.1):
    cu_idx = [i for i, s in enumerate(slab.get_chemical_symbols()) if s.upper() == 'CU']
    if not cu_idx:
        raise ValueError('slab 中无 Cu 原子')
    cu_pos = slab.get_positions()[cu_idx]
    zvals = sorted({round(float(z), 6) for z in cu_pos[:, 2]}, reverse=True)
    merged = []
    for z in zvals:
        if not merged or abs(z - merged[-1]) > tol:
            merged.append(z)
    if layer_idx > len(merged):
        raise ValueError(f'layer_idx={layer_idx} 超出 Cu 层数 {len(merged)}')
    zt = merged[layer_idx - 1]
    pts = cu_pos[np.abs(cu_pos[:, 2] - zt) <= tol]
    return pts


def estimate_two_shortest_dirs(slab: Atoms, anchor_xy, layer_idx=1):
    pts = get_layer_cu_positions(slab, layer_idx)
    xy = pts[:, :2]

    d = np.linalg.norm(xy - anchor_xy, axis=1)
    p0 = xy[np.argmin(d)]

    vecs = xy - p0
    norms = np.linalg.norm(vecs, axis=1)
    mask = norms > 1e-6
    vecs = vecs[mask]
    norms = norms[mask]

    order = np.argsort(norms)
    vec1 = None
    vec2 = None
    for idx in order:
        v = vecs[idx]
        if vec1 is None:
            vec1 = v
            continue
        # 选与 vec1 方向不共线的最近向量
        cosv = abs(np.dot(vec1, v) / (np.linalg.norm(vec1) * np.linalg.norm(v)))
        if cosv < 0.9:
            vec2 = v
            break

    if vec1 is None or vec2 is None:
        raise RuntimeError('无法估计两条最小路径方向')

    return vec1, vec2


def build_double_candidate(single1: Atoms, single2: Atoms, species1: str, species2: str, n_fixed: int, dir_vec_xy):
    n1 = len(get_species_molecule(species1)[0])
    n2 = len(get_species_molecule(species2)[0])

    slab = single1[:n_fixed].copy()
    ads1 = single1[n_fixed:n_fixed + n1].copy()
    ads2_template = single2[n_fixed:n_fixed + n2].copy()

    p1 = ads1.get_positions()[0]  # species1 anchor
    p2 = ads2_template.get_positions()[0]  # species2 anchor (template)

    target_anchor = np.array([p1[0] + dir_vec_xy[0], p1[1] + dir_vec_xy[1], p2[2]])
    shift = target_anchor - p2
    ads2_template.set_positions(ads2_template.get_positions() + shift)

    out = slab.copy()
    out.extend(ads1)
    out.extend(ads2_template)
    return out


f1 = find_single_best_file(single_final_root, species_1, single_best_file_1)
f2 = find_single_best_file(single_final_root, species_2, single_best_file_2)
print('single best #1:', f1)
print('single best #2:', f2)

single1 = read(str(f1), format='vasp')
single2 = read(str(f2), format='vasp')

# 方向估计基于 species_1 anchor 与第一层中心附近 Cu 最近邻
anchor_xy = single1[n_fixed].position[:2]
v1, v2 = estimate_two_shortest_dirs(single1[:n_fixed], anchor_xy, layer_idx=center_layer_from_top)
v1 = v1 * direction_distance_scale
v2 = v2 * direction_distance_scale

cand1 = build_double_candidate(single1, single2, species_1, species_2, n_fixed, v1)
cand2 = build_double_candidate(single1, single2, species_1, species_2, n_fixed, v2)

# 输出目录
if parent_folder_name == 'GO_ROOT':
    parent_dir = go_root
else:
    parent_dir = go_root.parent / parent_folder_name
parent_dir.mkdir(parents=True, exist_ok=True)

out1 = parent_dir / f'job_double_A_{species_1}__B_{species_2}__dir1.vasp'
out2 = parent_dir / f'job_double_A_{species_1}__B_{species_2}__dir2.vasp'
write(str(out1), cand1, format='vasp', vasp5=True)
write(str(out2), cand2, format='vasp', vasp5=True)
print('✅ 输出:')
print(' -', out1)
print(' -', out2)

if export_atom_index:
    for p, atoms in [(out1, cand1), (out2, cand2)]:
        idx_path = p.with_suffix('').with_name(p.stem + '_atom_index.txt')
        sy = atoms.get_chemical_symbols()
        pos = atoms.get_positions()
        with open(idx_path, 'w', encoding='utf-8') as f:
            f.write('# idx symbol x y z\n')
            for i, (s, r) in enumerate(zip(sy, pos), start=1):
                f.write(f'{i:4d} {s:>3s} {r[0]:12.6f} {r[1]:12.6f} {r[2]:12.6f}\n')
        print(' -', idx_path)

if package_vasp:
    vasp_files = sorted(parent_dir.glob('*.vasp'))
    if not package_all_vasp:
        vasp_files = [vf for vf in vasp_files if any(fnmatch(vf.name, pat) for pat in package_patterns)]
    if vasp_files:
        tar_name = package_name or f'{parent_dir.name}_double_vasp.tar.gz'
        tar_path = parent_dir / tar_name
        with tarfile.open(tar_path, 'w:gz') as tar:
            for vf in vasp_files:
                tar.add(vf, arcname=vf.name)
        print(f'📦 打包完成: {tar_path} ({len(vasp_files)} files)')
    else:
        print('⚠ 无可打包 .vasp 文件')

## 输出说明
- 输入来自 `1_body/Final` 的单吸附 best `.vasp`
- 输出双吸附候选：`job_double_A_<1>__B_<2>__dir1/dir2.vasp`
- 两个方向由表面最短路径方向自动估计
- 仅结构生成，不做计算提交
